# OmniCare Clinical Copilot - Notebook 3: Radiology & Imaging

**Pipeline:** DICOM/Medical Images → Orthanc Server → MedGemma Multimodal → Radiology Report

**Prerequisites:**
- Run `00_setup_and_models.ipynb` (models loaded)
- Run `01_consultation_audio_soap.ipynb` and `02_admission_vitals_fhir.ipynb`
- MedGemma 1.5-4b-it supports image+text input (radiology, dermatology, pathology, ophthalmology)

## 1. Setup & Configuration

In [ ]:
import os
import sys
import json
import torch
import numpy as np
from PIL import Image

sys.path.insert(0, os.path.dirname(os.path.abspath("__file__")))

from utils.encounter_state import load_encounter, update_stage, list_encounters
from utils.dicom_helpers import read_dicom_image, load_medical_image, download_sample_chest_xray
from utils.mcp_tools import analyze_medical_image

# Configuration
USE_ORTHANC = False  # Set to True if Orthanc is deployed
ORTHANC_URL = ""  # Fill in after deploying Orthanc on Cloud Run

# Load encounter
available = list_encounters()
encounter_id = available[-1] if available else None
print(f"Using encounter: {encounter_id}")

# Verify MedGemma is loaded
try:
    _ = medgemma_model.device
    print(f"MedGemma: loaded on {medgemma_model.device}")
except NameError:
    print("ERROR: MedGemma not loaded. Run Notebook 0 first!")
    raise

## 2. Deploy Orthanc DICOM Server (GCP Cloud Run)

Run this once to deploy the DICOM server.

In [ ]:
# Uncomment and run to deploy Orthanc on GCP Cloud Run
#
# !gcloud run deploy orthanc-dicom \
#   --image orthancteam/orthanc:latest \
#   --platform managed \
#   --region us-central1 \
#   --allow-unauthenticated \
#   --port 8042 \
#   --memory 1Gi \
#   --cpu 1 \
#   --min-instances 0 \
#   --max-instances 1
#
# !gcloud run services describe orthanc-dicom --region us-central1 --format 'value(status.url)'

print("Orthanc DICOM deployment instructions ready.")
print("After deploying, set ORTHANC_URL in cell 1.")

## 3. Obtain Medical Images

Options:
- Download public chest X-ray samples
- Upload your own DICOM files
- Use pydicom built-in test data

In [ ]:
# Install pydicom for DICOM handling
!pip install -q pydicom

# Option A: Download a sample chest X-ray (public dataset)
sample_image_path = download_sample_chest_xray("/content/sample_data")
print(f"Sample image: {sample_image_path}")

In [ ]:
# Option B: Use pydicom built-in test DICOM file
try:
    import pydicom
    from pydicom.data import get_testfiles_name

    # pydicom ships with sample DICOM files
    test_dicom = get_testfiles_name("CT_small.dcm")
    print(f"Built-in DICOM test file: {test_dicom}")

    ds = pydicom.dcmread(test_dicom)
    print(f"  Modality: {getattr(ds, 'Modality', 'N/A')}")
    print(f"  Rows x Cols: {ds.Rows} x {ds.Columns}")
    print(f"  Patient: {getattr(ds, 'PatientName', 'N/A')}")
except Exception as e:
    print(f"pydicom test file not available: {e}")
    test_dicom = None

In [ ]:
# Option C: Upload your own files
# from google.colab import files
# uploaded = files.upload()
# custom_image_path = list(uploaded.keys())[0]

## 4. Load and Display Medical Image

In [ ]:
# Load the medical image
image_path = sample_image_path  # Change to test_dicom or custom path as needed

image, metadata = load_medical_image(image_path)

print("Image Metadata:")
for k, v in metadata.items():
    print(f"  {k}: {v}")

print(f"\nImage size: {image.size}")
print(f"Image mode: {image.mode}")

# Display the image
from IPython.display import display
display(image.resize((400, 400)))

## 5. Upload to Orthanc (Optional)

In [ ]:
orthanc_id = None

if USE_ORTHANC and ORTHANC_URL and image_path.lower().endswith((".dcm", ".dicom")):
    from utils.dicom_helpers import upload_to_orthanc
    print(f"Uploading to Orthanc: {ORTHANC_URL}")
    try:
        result = upload_to_orthanc(image_path, ORTHANC_URL)
        orthanc_id = result.get("ID", "unknown")
        print(f"Uploaded successfully. Orthanc ID: {orthanc_id}")
    except Exception as e:
        print(f"Orthanc upload error: {e}")
else:
    print("Skipping Orthanc upload (using local image processing).")

## 6. MedGemma Multimodal Analysis

MedGemma 1.5-4b-it is specifically trained on:
- Chest X-rays
- Dermatology images
- Pathology slides
- Ophthalmology (fundus) images

In [ ]:
# Build clinical context from previous stages
encounter = load_encounter(encounter_id)
soap = encounter["stages"]["consultation"].get("soap_note", {})
admission = encounter["stages"]["admission"].get("admission_note", "")

clinical_context = ""
if soap:
    clinical_context += f"Assessment: {soap.get('assessment', 'N/A')}\n"
    clinical_context += f"Plan: {soap.get('plan', 'N/A')}\n"
if admission:
    clinical_context += f"\nAdmission Note Summary:\n{admission[:500]}\n"

if not clinical_context:
    clinical_context = "Patient presenting with productive cough, fever. Rule out pneumonia."

print("Clinical context for radiology:")
print(clinical_context[:500])

In [ ]:
# Run MedGemma multimodal analysis
print("Analyzing medical image with MedGemma (multimodal)...")

modality = metadata.get("modality", "X-ray")
body_part = metadata.get("body_part", "Chest")

radiology_report = analyze_medical_image(
    image=image,
    clinical_context=clinical_context,
    modality=modality,
    body_part=body_part,
    model=medgemma_model,
    processor=medgemma_processor,
    max_new_tokens=1024
)

print("\n" + "="*60)
print("RADIOLOGY REPORT")
print("="*60)
print(radiology_report)

## 7. Save to Encounter State

In [ ]:
encounter = update_stage(encounter_id, "radiology", {
    "images": [{
        "orthanc_id": orthanc_id or "local",
        "modality": modality,
        "body_part": body_part,
        "path": image_path
    }],
    "reports": [{
        "findings": radiology_report,
        "impression": "",  # Could parse from report if structured
        "recommendations": ""
    }]
})

print(f"Encounter {encounter_id} updated with radiology data.")
print(f"\nRadiology stage complete!")
print(f"Next: Run 04_discharge_summary.ipynb with encounter_id = '{encounter_id}'")

## 8. (Optional) Analyze Additional Images

You can analyze multiple images and append reports to the same encounter.

In [ ]:
# To analyze another image, repeat the process:
#
# image2, metadata2 = load_medical_image("path/to/another_image.dcm")
# report2 = analyze_medical_image(
#     image=image2, clinical_context=clinical_context,
#     modality=metadata2["modality"], body_part=metadata2["body_part"],
#     model=medgemma_model, processor=medgemma_processor
# )
#
# Then append to encounter state:
# enc = load_encounter(encounter_id)
# enc["stages"]["radiology"]["images"].append({...})
# enc["stages"]["radiology"]["reports"].append({...})
# save_encounter(enc)

print("Additional image analysis can be added as shown above.")